In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

import sys
import json
# -----------------------------
# Ajouter le repo au Python Path
# -----------------------------
sys.path.append("/Workspace/Users/mandu543@gmail.com/databricks_flights")

from src.Communs.utils import *
from conf.config import *

# 🔹 Avantages du nouveau notebook par rapport à l'ancien
## 1️⃣ Sécurité accrue
- L'ancien notebook utilisait `eval()` pour convertir la liste de colonnes clés → risque d’injection de code.
- Le nouveau notebook utilise `json.loads()` → sécurisé et robuste.
## 2️⃣ Lisibilité et maintenabilité
- Ancien notebook : SQL dynamique, temp views, jointures complexes → difficile à lire et maintenir.
- Nouveau notebook : DataFrame API, plus lisible, moins d’erreurs de syntaxe.
## 3️⃣ Modularité et réutilisabilité
- Ancien notebook : code spécifique à une table, peu réutilisable.
- Nouveau notebook : fonction `incremental_load()` → peut traiter n’importe quelle table Silver → Gold.
## 4️⃣ Gestion robuste des clés et CDC
- Surrogate Key (`DimSurrogateKey`) et dates (`create_date`, `update_date`) gérées correctement pour **anciennes et nouvelles lignes**.
- Merge Delta conditionné à l’existence de la table → pas de duplication de code.
## 5️⃣ Moins de code et plus performant
- Suppression des SQL dynamiques et temp views inutiles.
- Old/new records et union réalisés directement sur DataFrame → plus rapide et clair.
## 6️⃣ Extensible
- Possibilité d’ajouter facilement :
 - Backdated refresh global
 - Logging métriques (nb lignes insérées/mises à jour)
 - Ingestion multi-tables en batch

## Créer une alimentation Dynamique vers Gold Layer

#### Parameters

In [0]:
print("Initialisation des widgets...")

dbutils.widgets.text("key_cols_list", "")
dbutils.widgets.text("source_object", "")
dbutils.widgets.text("target_object", "")
dbutils.widgets.text("back_dated_refresh", "")

In [0]:
key_cols_list = dbutils.widgets.get("key_cols_list")
source_object = dbutils.widgets.get("source_object")
target_object = dbutils.widgets.get("target_object")
back_dated_refresh = dbutils.widgets.get("back_dated_refresh")

print("Parametres recuperes :")
print(f"  source_object       : {source_object}")
print(f"  target_object       : {target_object}")
print(f"  key_cols_list (raw) : {key_cols_list}")
print(f"  back_dated_refresh  : {back_dated_refresh}")

In [0]:
print(json.loads(key_cols_list))

In [0]:
if not key_cols_list:
   print("ERREUR : key_cols_list est vide")
   dbutils.notebook.exit("Key Cols List is Empty")
key_cols_list = json.loads(key_cols_list)

print(f"Colonnes cles converties : {key_cols_list}")
print(f"Nombre de colonnes cles : {len(key_cols_list)}")

#### Configuration

In [0]:
source_schema = SILVER_ZONE
target_schema   = GOLD_ZONE
cdc_column  = "modifiedDate"
colSurrogateKey = "DimSurrogateKey"

print("Configuration :")
print(f"  SILVER SCHEMA : {source_schema}")
print(f"  GOLD SCHEMA   : {target_schema}")
print(f"  CDC Column    : {cdc_column}")
print(f"  Surrogate Key : {colSurrogateKey}")

#### Fonction incrementale

In [0]:
def incremental_load(
   source_schema,
   source_table,
   target_schema,
   target_table,
   key_cols,
   cdc_col,
   surrogate_col,
   backdated_refresh=None
):
    print("\n==========================================")
    print("DEBUT DU PROCESSUS INCREMENTAL")
    print("==========================================")
    print(f"Source : {source_schema}.{source_table}")
    print(f"Target : {target_schema}.{target_table}")

    # 1 Last Load
    print("\nDetermination du LAST LOAD...")
    if backdated_refresh:
        last_load = backdated_refresh
        print("Backdated refresh detecte")
    elif spark.catalog.tableExists(f"{target_schema}.{target_table}"):
        print("Table cible existante detectee")
        last_load = spark.table(f"{target_schema}.{target_table}") \
                        .agg(max(cdc_col)) \
                        .first()[0]
    else:
        print("Table cible inexistante -> initial load")
        last_load = "1900-01-01 00:00:00"
    print(f"Last Load retenu : {last_load}")

    # 2 Lecture source
    print("\nLecture des donnees source incrementales...")
    df_source = spark.table(f"{source_schema}.{source_table}") \
                    .filter(col(cdc_col) >= last_load)
    source_count = df_source.count()
    print(f"Nombre de lignes lues depuis la source : {source_count}")
    if source_count == 0:
        print("Aucune donnee a traiter. Fin du notebook.")
        return

    # 3 Lecture cible
    print("\nVerification existence table cible...")
    if spark.catalog.tableExists(f"{target_schema}.{target_table}"):
        print("Table cible existante")
        df_target = spark.table(f"{target_schema}.{target_table}")
        target_count = df_target.count()
        print(f"Nombre de lignes actuelles en Gold : {target_count}")
        max_key = df_target.agg(max(surrogate_col)).first()[0] or 0
        print(f"Max Surrogate Key actuel : {max_key}")
    else:
        print("Table cible inexistante -> creation initiale")
        df_target = None
        max_key = 0

    # 4 Separation OLD / NEW
    print("\nSeparation OLD / NEW records...")
    if df_target:
        # df_source.createOrReplaceTempView("src")
        # df_target.createOrReplaceTempView("target")
        # join_condition = ' AND '.join([f"src.{i} = target.{i}" for i in key_cols_list]) # Join Condition
        # df_join = spark.sql(f"""
        #                 SELECT src.*,
        #                         target.{colSurrogateKey}, 
        #                         target.create_date, 
        #                         target.update_date 
        #                 FROM src 
        #                 LEFT JOIN target 
        #                 ON {join_condition} 
        #             """)
        
        # Colonnes clés pour la jointure
        join_condition = key_cols_list
        # Colonnes à récupérer depuis la table cible
        target_cols = [colSurrogateKey, "create_date", "update_date"]
        # Lorsque 'on' est une liste de noms de colonnes identiques des deux côtés,
        # Spark gère la condition de jointure automatiquement — pas besoin de vues ni de SQL.
        # Jointure LEFT entre source et cible sur les colonnes clés
        df_join = df_source.join(
                        df_target.select(*key_cols_list, *target_cols),
                        on=join_condition,
                        how="left"
        )

        print("Jointure effectuee")
 
        # Lignes existantes à mettre à jour (surrogate key présente)
        df_old = df_join.filter(col(surrogate_col).isNotNull()) \
                        .withColumn("update_date", current_timestamp())

        # Nouvelles lignes à insérer (surrogate key absente)
        # row_number() garantit des IDs continus et séquentiels à partir de max_key + 1
        df_new = df_join.filter(col(surrogate_col).isNull()) \
                            .withColumn(
                            surrogate_col,
                            row_number().over(Window.orderBy(key_cols_list)) + max_key) \
                            .withColumn("create_date", current_timestamp()) \
                            .withColumn("update_date", current_timestamp())
 
        old_count = df_old.count()
        new_count = df_new.count()
        print(f"OLD records detectes : {old_count}")
        print(f"NEW records detectes : {new_count}")
        df_union = df_old.unionByName(df_new)
        
        display(df_union)
    else:
        df_union = df_source \
        .withColumn(
            surrogate_col,
            row_number().over(Window.orderBy(key_cols_list))
        ) \
        .withColumn("create_date", current_timestamp()) \
        .withColumn("update_date", current_timestamp())
        print(f"Total records inseres : {df_union.count()}")


    # 5 Merge / Write
    print("\nEcriture en GOLD...")
    if spark.catalog.tableExists(f"{target_schema}.{target_table}"):
        print("Lancement du MERGE Delta...")
        delta_table = DeltaTable.forName(
            spark,
            f"{target_schema}.{target_table}"
        )
        delta_table.alias("target") \
            .merge(
                df_union.alias("source"),
                f"target.{surrogate_col} = source.{surrogate_col}"
            ) \
            .whenMatchedUpdateAll(
                condition=f"source.{cdc_col} >= target.{cdc_col}"
            ) \
            .whenNotMatchedInsertAll() \
            .execute()
        print("MERGE termine avec succes")
    else:
        print("Creation de la table Gold...")
        df_union.write.format("delta") \
            .mode("append") \
            .saveAsTable(f"{target_schema}.{target_table}")
        print("Table creee et donnees inserees")

    print("\n==========================================")
    print("FIN DU PROCESSUS INCREMENTAL")
    print("==========================================\n")

#### Execution



In [0]:
incremental_load(
   source_schema=SILVER_ZONE,
   source_table=source_object,
   target_schema=GOLD_ZONE,
   target_table=target_object,
   key_cols=key_cols_list,
   cdc_col=cdc_column,
   surrogate_col=colSurrogateKey,
   backdated_refresh=back_dated_refresh if back_dated_refresh else None
)
print("NOTEBOOK TERMINE AVEC SUCCES")